In [ ]:
import functools
import json
import os
import random
from pathlib import Path

from gossiplearning.config import Config
from gossiplearning.weight import weight_by_dataset_size
from utils.data import get_common_test_set
from utils.gossip_training import (
    get_node_dataset,
    round_trip_fn,
    run_simulation,
)
from utils.model_creators import create_LSTM

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

In [ ]:
prefix = "porto"
ds_name = "4in_notscaled"
n = 10
k = 3
datasets_folder = Path(f"data/datasets/porto_{n}n_{k}k")
networks_folder = Path(f"data/networks/porto_{n}n_{k}k")
network_variations = 10
failure_probabilites = [0.01, 0.005, 0.0025]

In [ ]:
def model_transmission_fn(i: int, j: int) -> int:
    return random.randint(25, 35)

In [ ]:
with open("failures_simulation_config.json", "r") as f:
    config_json = json.load(f)
    config = Config.model_validate(config_json)

In [ ]:
model_creator = functools.partial(create_LSTM, config=config)

In [ ]:
workspace_dir_prefix = config.workspace_dir
for failure_probability in failure_probabilites:
    config.training.node_failure_probability = failure_probability
    config.workspace_dir = f"{workspace_dir_prefix}/{config.training.node_failure_probability}"
    for sim_num in range(network_variations):
        all_history_plot = (
            Path(config.workspace_dir) / str(sim_num) / "plots" / "history" / "all.jpg"
        )
        if all_history_plot.exists():
            continue
    
        node_data_fn = functools.partial(
                get_node_dataset,
                base_folder=datasets_folder,
                simulation_number=sim_num,
                ds_name=ds_name,
            )

        get_test_set = functools.partial(
            get_common_test_set,
            node_data_fn=node_data_fn,
            n_nodes=config.n_nodes,
            perc=0.1,
        )

        run_simulation(
            config=config,
            simulation_number=sim_num,
            network_folder=networks_folder / str(sim_num),
            round_trip_fn=round_trip_fn,
            model_transmission_fn=model_transmission_fn,
            node_data_fn=node_data_fn,
            model_creator=model_creator,
            get_test_set=get_test_set,
            weight_fn=weight_by_dataset_size,
        )